# Subqueries

## Introduction

As queries grow more complex, it helps to break them into smaller, composable parts. A subquery is a `SELECT` statement nested inside another query. Subqueries can appear in the `WHERE`, `FROM`, or `SELECT` clause and let you filter or compute based on the result of another query.

## Objectives

You will be able to:

- Write subqueries in the `WHERE` clause using `IN`
- Use subqueries in the `FROM` clause as a derived table
- Decompose queries that would otherwise require complex joins
- Combine subqueries with aggregates and `HAVING`

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('data/sql_subqueries/data.sqlite')
cur = conn.cursor()

CRM schema for reference:

![CRM schema](assets/sql_subqueries/Database-Schema.png)

---

## Subqueries in WHERE with IN

The `IN` operator checks whether a value appears in a list. When that list is computed from another `SELECT`, you get a subquery.

**Example:** find all employees who work in a US office.

With a join:

In [ ]:
cur.execute("""
    SELECT lastName, firstName, officeCode
    FROM employees
    JOIN offices USING(officeCode)
    WHERE country = 'USA';
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

With a subquery — equivalent result, no explicit join:

In [ ]:
cur.execute("""
    SELECT lastName, firstName, officeCode
    FROM employees
    WHERE officeCode IN (
        SELECT officeCode
        FROM offices
        WHERE country = 'USA'
    );
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

---

## Subqueries for Aggregate-Based Filters

Some filters can't be expressed with `WHERE` because they depend on an aggregate. A subquery handles this cleanly.

**Example:** find all employees who work in offices with at least 5 employees.

In [ ]:
cur.execute("""
    SELECT lastName, firstName, officeCode
    FROM employees
    WHERE officeCode IN (
        SELECT officeCode
        FROM offices
        JOIN employees USING(officeCode)
        GROUP BY 1
        HAVING COUNT(employeeNumber) >= 5
    );
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

---

## Subqueries in FROM (Derived Tables)

A subquery in the `FROM` clause produces a temporary result set — a derived table — that the outer query treats like a regular table. This is useful when you want to aggregate an already-aggregated result.

**Example:** average of each customer's average payment.

In [ ]:
cur.execute("""
    SELECT AVG(customerAvgPayment) AS overall_avg_payment
    FROM (
        SELECT AVG(amount) AS customerAvgPayment
        FROM payments
        JOIN customers USING(customerNumber)
        GROUP BY customerNumber
    );
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

---

## Practice

Use the CRM database for all exercises.

In [ ]:
pconn = sqlite3.connect('data/sql_subqueries_lab/data.sqlite')
pc = pconn.cursor()

**Q1.** Rewrite the following join query using a subquery instead — no `JOIN` is needed in your answer.

```sql
SELECT customerNumber, contactLastName, contactFirstName
FROM customers
JOIN orders USING(customerNumber)
WHERE orderDate = '2003-01-31';
```

In [ ]:
# Your code here


**Q2.** Select the product name and total number of orders for each product, sorted by total orders descending.

In [ ]:
# Your code here


**Q3.** Select each product name and the total number of **distinct customers** who have ordered it, sorted descending.

> Hint: use `COUNT(DISTINCT customerNumber)` to avoid double-counting customers who placed multiple orders.

In [ ]:
# Your code here


**Q4.** Select the employee number, first name, last name, office city, and office code of employees who sold products that were ordered by fewer than 20 distinct customers. List each employee only once.

In [ ]:
# Your code here


**Q5.** Select the employee number, first name, last name, and customer count for employees whose customers have an average credit limit above $15,000.

In [ ]:
# Your code here


---

## Summary

In this notebook you learned how to:

- Use `WHERE col IN (SELECT ...)` to filter rows based on another query's result
- Combine subqueries with `HAVING` to filter on aggregates without fetching aggregate data
- Use a subquery in `FROM` as a derived table to compute "aggregates of aggregates"
- Choose between a join and a subquery based on what the query is asking for

Next: [06 — SQL with pandas](06_sql_with_pandas.ipynb)